# 01 · Exploring TorNet

**Goal:** understand the data before training anything.

1. Read `catalog.csv` (all 203,133 samples, 38 MB) and count samples and tornadoes per year, **without downloading the radar data**.
2. Check the official train/test split for storm (episode) overlap.
3. Look at real radar images: a tornado, a warning without a tornado and a random storm.
4. Check that our preprocessing keeps what matters.

Runs on Colab (CPU is enough) or locally. Only a few hundred MB are downloaded.

In [ ]:
# --- Setup: works on Google Colab and on your own computer ---
import os, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")                     # processed data is kept on Drive
    REPO = Path("/content/tornado-cnn")
    if not REPO.exists():
        token = userdata.get("GITHUB_TOKEN")          # Colab Secrets (key icon on the left)
        !git clone -q https://{token}@github.com/baevv/tornado-cnn.git {REPO}
    else:
        !git -C {REPO} pull -q
    %pip install -q h5netcdf h5py
    os.chdir(REPO)
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(REPO))
print("Colab" if IN_COLAB else "Local", "| repo:", REPO)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import paths, tornet, viz

cfg = paths.load_config()
DATA = paths.data_dir()
print("data dir:", DATA)
print("config:", cfg)

## 1. The catalog

Each row is one sample. `category` has three values:

| category | meaning | our label |
|---|---|---|
| `TOR` | confirmed tornado | **1** |
| `WRN` | tornado warning was issued, but no tornado | 0 |
| `NUL` | random storm, no tornado | 0 |

`WRN` samples are the hard negatives: storms that looked dangerous enough for forecasters to warn.

In [ ]:
cat = tornet.get_catalog(DATA)
print(f"{len(cat):,} samples")
cat.head(3)

In [ ]:
per_year = pd.crosstab(cat["year"], cat["category"])
per_year["total"] = per_year.sum(axis=1)
per_year["tornado_%"] = (100 * per_year["TOR"] / per_year["total"]).round(1)
sizes_gb = {2013: 3.2, 2014: 15.1, 2015: 17.4, 2016: 16.0, 2017: 15.1,
            2018: 12.5, 2019: 18.2, 2020: 17.0, 2021: 18.0, 2022: 19.0}   # archive size on Zenodo
per_year["archive_GB"] = pd.Series(sizes_gb)
per_year["TOR_per_GB"] = (per_year["TOR"] / per_year["archive_GB"]).round(0)
per_year

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5), layout="constrained")
per_year["TOR"].plot.bar(ax=ax, color=["C1" if y in cfg["years"] else "C7" for y in per_year.index])
ax.set_ylabel("tornado samples")
ax.set_title("Tornado samples per year (orange = years used in this project)")
plt.show()

### Years used in this project

2013 is small (data starts end of August 2013). 2017 and 2019 give the most tornado samples per downloaded GB, 2015 adds more data.

In [ ]:
sel = cat[cat["year"].isin(cfg["years"])]
summary = pd.crosstab(sel["type"], sel["category"], margins=True)
print("Years:", cfg["years"], f"| {len(sel):,} samples | {sel['label'].sum():,} tornado ({100*sel['label'].mean():.1f}%)")
summary

### Tornado strength (EF scale)

EF0 is the weakest, EF4 among the strongest. We do not predict EF, but later we can check whether the model finds strong tornadoes more easily.

In [ ]:
tor = cat[cat["category"] == "TOR"]
pd.crosstab(tor["year"], tor["ef_number"].astype(int), margins=True)

## 2. Train/test split check

The dataset comes with a `type` column (`train` / `test`). If the same storm episode appears on both sides,
the model could be tested on a storm it has already seen (leakage). These episodes will be removed from training.

In [ ]:
train_eps = set(cat.loc[cat["type"] == "train", "episode_id"])
test_eps = set(cat.loc[cat["type"] == "test", "episode_id"])
overlap = train_eps & test_eps
print(f"test episodes: {len(test_eps):,} | also in train: {len(overlap)}")
print(f"train samples from those episodes: {cat[(cat['type']=='train') & cat['episode_id'].isin(overlap)].shape[0]}")

## 3. Looking at real radar samples

We read only the beginning of the 2017 archive and stop once we have a few samples of each category
(a few hundred MB instead of 15 GB).

What one sample contains (checked on the files):

* 6 radar variables, each with shape `(time=4, azimuth=120, range=240, sweep=2)`
* 4 frames, 5 minutes apart; 2 elevation angles (0.5° and 0.9°)
* a 60° × 60 km window around the storm
* missing measurements are `NaN`

We use two variables:

* **DBZ (reflectivity):** how much rain/hail is in the air. A tornadic storm often shows a **hook** shape.
* **VEL (radial velocity):** wind toward (blue) or away from (red) the radar. A tornado shows up as a
  **couplet**: strong blue and strong red right next to each other, meaning rotation.

In [ ]:
samples = tornet.sample_from_tar(tornet.tar_url(2017), per_category={"TOR": 2, "WRN": 1, "NUL": 1})
for name in samples:
    print(name)

In [ ]:
ds0 = next(iter(samples.values()))
ds0

In [ ]:
for name, ds in samples.items():
    viz.plot_sample(ds)
    plt.show()

### The 4 time frames of a tornado sample

`frame_labels` marks the frames where a tornado was on the ground. The sample label is the **last** frame.
The earlier frames will matter later (RNN/LSTM weeks).

In [ ]:
tor_name = next(n for n in samples if "/TOR_" in n)
viz.plot_frames(samples[tor_name], "VEL")
plt.show()
viz.plot_frames(samples[tor_name], "DBZ")
plt.show()

## 4. What the network will see

Preprocessing (see `src/tornet.py`):

1. Keep **DBZ and VEL**, the **lowest sweep (0.5°)** and the **last frame** → shape `(2, 120, 240)`.
2. Scale with **fixed physical ranges** (DBZ −20…60 dBZ, VEL −60…60 m/s), not with statistics of our data, so nothing leaks from the test set.
3. Store as `uint8` (1 byte per pixel, 4× smaller), with `0` reserved for missing pixels.
4. When loading, map to `[-1, 1]` and fill missing pixels with a physical value: DBZ → lowest value (no echo), VEL → 0 (no motion).

The network sees the polar grid directly (rows = azimuth, columns = range), so shapes look stretched compared to the map above.

In [ ]:
ds = samples[tor_name]
q = tornet.preprocess_sample(ds, cfg["variables"], cfg["sweep"], cfg["frame"])
x = tornet.from_uint8(q, cfg["variables"])
print("stored:", q.shape, q.dtype, "| network input:", x.shape, x.dtype, f"[{x.min():.2f}, {x.max():.2f}]")
print("missing pixels per channel (%):", (100 * (q == 0).mean(axis=(1, 2))).round(1))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), layout="constrained")
for ax, v, img in zip(axes, cfg["variables"], x):
    m = ax.imshow(img, aspect="auto", cmap=viz.CMAPS[v], vmin=-1, vmax=1)
    ax.set_title(f"{v} (network input)")
    ax.set_xlabel("range bin (250 m)")
    ax.set_ylabel("azimuth bin (0.5°)")
    plt.colorbar(m, ax=ax)
plt.show()

## Summary

* TorNet: 203,133 samples, 13,857 tornadoes (6.8%). Strongly imbalanced, so accuracy alone will be misleading.
* Years used: see the table above. Test samples come from each year's official `test` split, and overlapping episodes are removed from training.
* Next: `02_preprocess.ipynb` downloads the selected years on Colab and saves the compact arrays to Google Drive.